In [ ]:
import os
# TODO: Set your API key as environment variable before running
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'
os.environ['OPENAI_BASE_URL'] = 'https://llmapi.paratera.com/v1'
from tasks import load_logical7
from openai import OpenAI
import re
from tqdm import tqdm

In [2]:
def call_openai(prompt, model='Qwen3-30B-A3B-Instruct-2507'):
    client = OpenAI(
        api_key=os.environ['OPENAI_API_KEY'],
        base_url=os.environ['OPENAI_BASE_URL']
    )
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [3]:
def bbh_mcq_postprocess(text: str) -> str:
    ans = text
    ans_line = ans.split('answer is ')
    if len(ans_line) != 1:
        ans = ans_line[1].strip()
    match = re.search(r'\(([A-Z])\)*', ans)
    if match:
        return match.group(1)
    match = re.search(r'([A-Z])', ans)
    if match:
        return match.group(1)
    return ans

def bbh_mcq_eval_fn(prediction: str, ground_truth_answer: str):
    pred = bbh_mcq_postprocess(prediction)
    ref = bbh_mcq_postprocess(ground_truth_answer)
    return int(pred == ref)

In [4]:
train_data, eval_data, test_data = load_logical7()
initial_prompt = """You must give your final answer by starting with 'So the answer is'"""

In [5]:
error_idx = []
for i, sample in tqdm(enumerate(train_data)):
    prompt = f"{initial_prompt}\n{sample['input']}"
    raw_response = call_openai(prompt)
    is_correct = bbh_mcq_eval_fn(raw_response, sample['target'])
    if not is_correct:
        error_idx.append(i)

50it [11:09, 13.38s/it]


In [6]:
bad_cases = [train_data[i] for i in error_idx]

In [9]:
first_case = bad_cases[0]
rep = call_openai(f"{initial_prompt}\n{first_case['input']}")
print(rep)
print(bbh_mcq_postprocess(rep))
print(first_case['target'])
print(bbh_mcq_eval_fn(rep, first_case['target']))

We are given a set of seven golfers: Eve, Rob, Dan, Mel, Ana, Eli, and Ada. They finished in a fixed order from 1st (best) to 7th (worst). We are given several clues, and we must determine who finished first based on the logical implications.

Let’s list the clues:

1. **Ada finished above Rob.**  
   → Ada < Rob (in position number; lower number = better)

2. **Eve finished below Rob.**  
   → Eve > Rob

3. **Mel finished above Eli.**  
   → Mel < Eli

4. **Ada finished below Dan.**  
   → Ada > Dan

5. **Ana finished third.**  
   → Ana = 3rd

6. **Eli finished second.**  
   → Eli = 2nd

Now let’s use these step by step.

From (5):  
- Ana = 3rd

From (6):  
- Eli = 2nd

From (3): Mel finished above Eli → Mel < Eli  
Since Eli is 2nd, Mel must be 1st (only position above 2nd).

So:  
- Mel = 1st

So far:
1. Mel  
2. Eli  
3. Ana  
? ? ?

Now from (4): Ada finished below Dan → Ada > Dan  
So Dan finished before Ada.

From (1): Ada finished above Rob → Ada < Rob  
So Ada is ahead of R

In [8]:
bad_cases

[{'input': 'The following paragraphs each describe a set of seven objects arranged in a fixed order. The statements are logically consistent within each paragraph. In a golf tournament, there were seven golfers: Eve, Rob, Dan, Mel, Ana, Eli, and Ada. Ada finished above Rob. Eve finished below Rob. Mel finished above Eli. Ada finished below Dan. Ana finished third. Eli finished second.\nOptions:\n(A) Eve finished first\n(B) Rob finished first\n(C) Dan finished first\n(D) Mel finished first\n(E) Ana finished first\n(F) Eli finished first\n(G) Ada finished first',
  'target': '(D)'}]

In [ ]:
bbh_mcq_postprocess(rep)

In [ ]:
bbh_mcq_eval_fn(rep, first_case['target'])

In [11]:
print(len(rep.split(" ")))

613
